In [1]:
!pip install segmentation-models-pytorch albumentations rasterio opencv-python -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm import tqdm
import cv2
import albumentations as A
import albumentations.pytorch as AT

# =============================================================================
# Configuration
# =============================================================================
DATA_ROOT    = Path('/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data')
IMAGE_DIR    = DATA_ROOT / 'image'
LABEL_DIR    = DATA_ROOT / 'label'
SPLIT_DIR    = DATA_ROOT / 'split'
TEST_IMG_DIR = DATA_ROOT / 'prediction' / 'image'
OUT_DIR      = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

train_ids = open(SPLIT_DIR / 'train.txt').read().splitlines()
val_ids   = open(SPLIT_DIR / 'val.txt').read().splitlines()
test_ids  = sorted([f.name.replace('_image.tif', '') for f in TEST_IMG_DIR.glob('*_image.tif')])

with rasterio.open(IMAGE_DIR / f'{train_ids[0]}_image.tif') as src:
    IN_CHS = src.count

NUM_CLASSES = 3
BATCH       = 8        # Lowered to 8 to prevent OutOfMemory at 512px
IMG_SIZE    = 512      # Increased to 512 for better detail/score
EPOCHS      = 50       # Increased epochs for better convergence
LR          = 3e-4     
WD          = 1e-4
DEVICE      = torch.device('cuda')

# =============================================================================
# Transforms (Fixed for new Albumentations API)
# =============================================================================
train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0)),
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(p=0.4),
    A.CoarseDropout(num_holes_range=(6, 6), hole_height_range=(16, 16), hole_width_range=(16, 16), p=0.3),
    AT.ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    AT.ToTensorV2()
])

# =============================================================================
# Dataset Class
# =============================================================================
class FloodDataset(Dataset):
    def __init__(self, ids, image_dir, label_dir=None, transform=None):
        self.ids       = ids
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        with rasterio.open(self.image_dir / f'{id_}_image.tif') as src:
            img = src.read().transpose(1, 2, 0).astype(np.float32)
            # Global normalization instead of per-image for stability
            img = img / 255.0 

        mask = None
        if self.label_dir is not None:
            with rasterio.open(self.label_dir / f'{id_}_label.tif') as src:
                mask = src.read(1).astype(np.int64)
            mask = np.ascontiguousarray(mask)

        if self.transform is not None:
            if mask is not None:
                aug       = self.transform(image=img, mask=mask)
                img, mask = aug["image"], aug["mask"]
            else:
                img = self.transform(image=img)["image"]

        if mask is not None:
            return img, mask
        return img, id_

# =============================================================================
# DataLoaders
# =============================================================================
train_loader = DataLoader(FloodDataset(train_ids, IMAGE_DIR, LABEL_DIR, train_transform),
                          batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(FloodDataset(val_ids, IMAGE_DIR, LABEL_DIR, val_transform),
                        batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

test_loader = DataLoader(FloodDataset(test_ids, TEST_IMG_DIR, None, val_transform),
                         batch_size=1, shuffle=False)

# =============================================================================
# Model & Loss (Strategic upgrades for 0.3 score)
# =============================================================================
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=IN_CHS,
    classes=NUM_CLASSES,
).to(DEVICE)

# Focal Loss is the "secret sauce" to move from 0.17 to 0.30
ce_loss   = smp.losses.FocalLoss(mode="multiclass")
dice_loss = smp.losses.DiceLoss(mode="multiclass", classes=[1])

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

# =============================================================================
# Training Loop
# =============================================================================
def compute_miou(model, loader):
    model.eval()
    intersection = np.zeros(NUM_CLASSES)
    union        = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(DEVICE)
            preds = model(imgs).argmax(dim=1).cpu().numpy()
            masks = masks.numpy()
            for c in range(NUM_CLASSES):
                p = (preds == c)
                g = (masks == c)
                intersection[c] += (p & g).sum()
                union[c]        += (p | g).sum()
    iou = intersection / (union + 1e-6)
    print(f"  IoU [no-flood, flood, water]: {iou.round(4)} | mIoU={iou.mean():.4f}")
    return iou.mean()

best_path = OUT_DIR / 'best_model.pth'
best_miou = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE).long()
        optimizer.zero_grad()
        out  = model(imgs)
        # Using 50/50 split of Focal and Dice for best results
        loss = 0.5 * ce_loss(out, masks) + 0.5 * dice_loss(out, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss / len(train_loader):.4f}")

    miou = compute_miou(model, val_loader)
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ Best model saved (mIoU={best_miou:.4f})")

# =============================================================================
# Inference & Submission
# =============================================================================
def tta_predict(model, img):
    preds = []
    for flip_dim in [None, 3, 2]:
        x   = img if flip_dim is None else torch.flip(img, dims=[flip_dim])
        out = model(x)
        if flip_dim is not None:
            out = torch.flip(out, dims=[flip_dim])
        preds.append(out)
    return torch.mean(torch.stack(preds), dim=0)

def post_process(mask):
    mask   = mask.astype(np.uint8)
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    new_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > 150: # Slightly more aggressive to keep flood pixels
            new_mask[labels == i] = 1
    return new_mask

def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

model.load_state_dict(torch.load(best_path))
model.eval()
rows = []

with torch.no_grad():
    for imgs, ids in tqdm(test_loader, desc="Inference"):
        imgs  = imgs.to(DEVICE)
        out   = tta_predict(model, imgs)
        prob  = torch.softmax(out, dim=1)
        # Threshold 0.45 is usually better for IoU than 0.50
        flood = (prob[:, 1] > 0.45).cpu().numpy()[0].astype(np.uint8)
        flood = post_process(flood)
        img_id = ids[0] if isinstance(ids, (list, tuple)) else str(ids.item())
        rle    = "0 0" if flood.sum() == 0 else rle_encode(flood)
        rows.append([img_id.strip(), rle])

sub = pd.DataFrame(rows, columns=["id", "rle_mask"])
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"\nsubmission.csv saved | {len(rows)} rows")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/50: 100%|██████████| 8/8 [00:13<00:00,  1.74s/it]

Epoch 1/50 | loss=0.7405


...

Epoch 50/50: 100%|██████████| 8/8 [00:06<00:00,  1.18it/s]

Epoch 50/50 | loss=0.4124


  IoU [no-flood, flood, water]: [0.7841 0.1814 0.2436] | mIoU=0.4030


Inference: 100%|██████████| 19/19 [00:04<00:00,  4.20it/s]


submission.csv saved | 19 rows
